# 2. metadata filter 검색

**시나리오:** refund와 account라는 단어가 섞여도 billing으로 분류된 티켓에는 billing playbook만 검색합니다.

**학습 목표:** semantic 유사도 전에 category `filter`를 적용해 잘못된 근거 인용을 줄이는 방법을 익힙니다.

## 중요 변수·함수

- `SUPPORT_PLAYBOOKS[*].metadata["category"]`: 업무 영역 필터입니다.
- `retriever.search(..., filter={"category": ...})`: fixture와 PGVector가 공유하는 검색 계약입니다.
- `k`: 필터를 통과한 후보 중 가져올 최대 개수입니다.

In [ ]:
# 이 학습 Notebook은 외부 API/DB를 사용하지 않는 fixture 모드로 고정합니다.
import os
os.environ["APP_MODE"] = "fixture"

# Notebook 위치에서 실행해도 repository의 canonical app을 가져옵니다.
from pathlib import Path
import sys

_repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# 혼합 키워드 질문을 billing 영역으로 제한합니다.
from week2.app import SUPPORT_PLAYBOOKS, create_fixture_services

retriever = create_fixture_services().retriever
billing_docs = retriever.search('refund account error', k=5, filter={'category': 'billing'})
[(doc.metadata['category'], doc.metadata['chunk_id']) for doc in billing_docs]

In [ ]:
# 같은 질문이라도 access filter는 다른 업무 근거만 허용합니다.
access_docs = retriever.search('refund account error', k=5, filter={'category': 'access'})
assert all(doc.metadata['category'] == 'billing' for doc in billing_docs)
assert all(doc.metadata['category'] == 'access' for doc in access_docs)
{'billing': len(billing_docs), 'access': len(access_docs)}

## 예측 과제와 해석

**예측 과제:** filter 없이 lexical/embedding 유사도만 사용했을 때 발생할 수 있는 오인용을 설명하세요.

**해석:** metadata filter는 검색 대상을 업무 영역으로 먼저 제한합니다. Week 1의 기본 검색을 반복하는 것이 아니라 검색 품질을 통제하는 새 단계입니다.